<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
# <font color='blue'>Data Science Academy</font>
## <font color='blue'>Data Science Para Análise Multivariada de Dados</font>
## <font color='blue'>Projeto 9</font>
### <font color='blue'>Data Science no Agronegócio</font>
### <font color='blue'>Previsão de Rendimento de Colheita e Otimização da Irrigação</font>

## Instalando e Carregando os Pacotes
<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [ ]:
!pip install -q -U watermark

https://www.tensorflow.org/

In [ ]:
#!pip install tensorflow

In [ ]:
%env TF_CPP_MIN_LOG_LEVEL=3

In [ ]:
# Imports
import joblib
import sklearn
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%reload_ext watermark
%watermark -a "Data Science Academy"

## Carregando o Conjunto de Dados

In [ ]:
# Carregar o dataset
df_dsa = pd.read_csv('dataset.csv')

In [ ]:
df_dsa.shape

In [ ]:
df_dsa.head()

In [ ]:
df_dsa.tail()

## Análise Exploratória

In [ ]:
# Verificar os tipos de dados das colunas
df_dsa.dtypes

In [ ]:
# Exibir as colunas do dataset
df_dsa.columns

In [ ]:
# Identificar colunas não numéricas
non_numeric_columns = df_dsa.select_dtypes(include = ['object']).columns
print(f'Colunas não numéricas: {non_numeric_columns}')

## Limpeza e Transformação

In [ ]:
# Remover colunas não numéricas (se não forem necessárias)
df_dsa = df_dsa.drop(columns = non_numeric_columns)

In [ ]:
# Verificar se a coluna 'umidade' contém valores numéricos
if df_dsa['umidade'].dtype == 'object':
    df_dsa['umidade'] = pd.to_numeric(df_dsa['umidade'], errors = 'coerce')

In [ ]:
# Remover linhas com valores faltantes
df_dsa = df_dsa.dropna()

## Padronização dos Dados

In [ ]:
# Separar preditores e variável alvo
X = df_dsa.drop(columns = 'umidade')
y = df_dsa['umidade']

In [ ]:
# Dividir os dados em conjunto de treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
# Cria o padronizador
scaler = StandardScaler()

In [ ]:
# Padroniza os dados
X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

In [ ]:
# Salva o padronizador no disco
joblib.dump(scaler, 'scaler_dsa.joblib')

## Construção do Modelo

Acompanhe os detalhes no videobook do Capítulo 19.

In [ ]:
# Definir a arquitetura do modelo
modelo_dsa = Sequential([Dense(64, activation = 'relu', input_shape = (X_treino.shape[1],)),
                         Dropout(0.3),
                         Dense(32, activation = 'relu'),
                         Dropout(0.3),
                         Dense(16, activation = 'relu'),
                         Dense(1)])

O código acima define a arquitetura de um modelo de rede neural sequencial utilizando a biblioteca Keras. Aqui está uma explicação detalhada de cada linha:

**modelo_dsa = Sequential([ ... ])**: Cria um modelo sequencial, que é uma pilha linear de camadas.

**Dense(64, activation='relu', input_shape=(X_treino.shape[1],))**: Adiciona uma camada densa (totalmente conectada) com 64 neurônios e função de ativação ReLU (Rectified Linear Unit). A input_shape especifica a forma dos dados de entrada, que corresponde ao número de características (colunas) em X_treino.

**Dropout(0.3)**: Adiciona uma camada de dropout com uma taxa de 30%. Dropout é uma técnica de regularização que desativa aleatoriamente 30% dos neurônios durante o treinamento para prevenir o overfitting.

**Dense(32, activation='relu')**: Adiciona outra camada densa com 32 neurônios e função de ativação ReLU.

**Dropout(0.3)**: Adiciona mais uma camada de dropout com uma taxa de 30%.

**Dense(16, activation='relu')**: Adiciona mais uma camada densa com 16 neurônios e função de ativação ReLU.

**Dense(1)**: Adiciona a camada de saída com um único neurônio. Não é especificada uma função de ativação, o que é comum em problemas de regressão, onde a saída é um valor contínuo. Se fosse um problema de classificação binária, uma ativação sigmoid poderia ser usada nesta camada.

Essa arquitetura é típica para problemas de regressão onde se deseja prever um único valor contínuo a partir de múltiplas características de entrada.

## Compilação do Modelo

Acompanhe os detalhes no videobook do Capítulo 19.

In [ ]:
# Compilar o modelo
modelo_dsa.compile(optimizer = 'adam', loss = 'mse', metrics = ['mae'])

O código acima compila o modelo de rede neural que foi definido anteriormente. A compilação é um passo necessário antes de treinar o modelo. Aqui está uma explicação de cada parte do código:

**modelo_dsa.compile(...)**: Compila o modelo, configurando-o para treinamento com um otimizador, uma função de perda e métricas.

**optimizer='adam'**: Define o otimizador como 'adam'. O Adam (Adaptive Moment Estimation) é um otimizador eficiente e amplamente utilizado em redes neurais, combinando as vantagens dos algoritmos de Gradient Descent com Momentum e RMSProp. Ele ajusta dinamicamente a taxa de aprendizado durante o treinamento.

**loss='mse'**: Define a função de perda como 'mse' (Mean Squared Error). MSE é uma função de perda comum em problemas de regressão, que calcula a média dos quadrados das diferenças entre os valores previstos e os valores reais. É usada para medir o desempenho do modelo, com valores menores indicando um melhor ajuste.

**metrics=['mae']**: Especifica que a métrica a ser monitorada durante o treinamento é 'mae' (Mean Absolute Error). MAE é a média dos valores absolutos das diferenças entre os valores previstos e os valores reais. Assim como MSE, valores menores de MAE indicam um melhor desempenho do modelo, mas MAE é menos sensível a grandes erros do que MSE.

Compilar o modelo é um passo essencial que define como o modelo será otimizado e avaliado durante o treinamento.

## Definindo os Callbacks

Acompanhe os detalhes no videobook do Capítulo 19.

In [ ]:
# Callbacks
early_stopping = EarlyStopping(monitor = 'val_loss', patience = 10, restore_best_weights = True)
model_checkpoint = ModelCheckpoint('modelo_dsa.keras', save_best_only = True)

Este trecho de código define dois callbacks para o treinamento do modelo: EarlyStopping e ModelCheckpoint. Callbacks são ferramentas que permitem realizar certas ações em pontos específicos durante o treinamento.

**early_stopping = EarlyStopping(...)**: Define um callback de parada antecipada (early stopping).

**monitor='val_loss'**: Monitora a perda de validação (val_loss) durante o treinamento. A validação é um processo onde um conjunto separado de dados (dados de validação) é usado para avaliar o desempenho do modelo, ajudando a evitar o overfitting.

**patience=10**: Define a paciência em 10 épocas. Isso significa que se a perda de validação não melhorar por 10 épocas consecutivas, o treinamento será interrompido.

**restore_best_weights=True**: Restabelece os pesos do modelo para o estado em que apresentaram a melhor perda de validação. Isso garante que os pesos finais do modelo sejam os melhores encontrados durante o treinamento.

**model_checkpoint = ModelCheckpoint(...)**: Define um callback de checkpoint do modelo.

**'modelo_dsa.keras'**: Especifica o nome do arquivo onde o modelo será salvo.

**save_best_only=True**: Salva o modelo apenas quando ele apresentar a melhor perda de validação até o momento. Isso evita salvar várias versões do modelo e garante que o melhor modelo encontrado durante o treinamento seja salvo.

Em resumo, esses callbacks ajudam a:

- Parar o treinamento antecipadamente se o modelo não estiver melhorando, economizando tempo e recursos computacionais.

- Salvar automaticamente o melhor modelo encontrado durante o treinamento, garantindo que você tenha uma versão do modelo com o melhor desempenho.

## Treinamento do Modelo

In [ ]:
modelo_dsa.summary()

In [ ]:
# Treinar o modelo
history = modelo_dsa.fit(X_treino_scaled, 
                         y_treino,
                         validation_split = 0.2,
                         epochs = 100,
                         batch_size = 32,
                         callbacks = [early_stopping, model_checkpoint])

## Avaliação do Modelo

In [ ]:
# Avaliar o modelo no conjunto de teste
teste_loss, teste_mae = modelo_dsa.evaluate(X_teste_scaled, y_teste)

In [ ]:
print(f'Teste Loss: {teste_loss}')
print(f'Teste MAE: {teste_mae}')

Vamos agora fazer o deploy do modelo via API no próximo capítulo.

In [ ]:
%reload_ext watermark
%watermark -a "Data Science Academy"

In [ ]:
#%watermark -v -m

In [ ]:
#%watermark --iversions

# Fim